# Redes Neuronales 2026

---

# Guía de ejercicios N°9: *Perceptron simple* y *Perceptron* con una capa oculta
# Una neurona con dos entradas y `bias` estándar de PyTorch
## Implementación con PyTorch y `nn.Sequential`

En esta notebook utilizaremos una neurona con dos entradas:

$$
\mathbf{x}=(x_1,x_2),
$$

y el `bias` estándar de `nn.Linear`:

$$
h=w_1x_1+w_2x_2+b.
$$

Luego aplicaremos una función *sigmoide*:

$$
\hat y=\sigma(h)=\frac{1}{1+e^{-h}}.
$$

La primera parte contiene un **ejemplo completo resuelto para la compuerta OR**.

A partir de ese ejemplo, deberá repetir el procedimiento para:

1. intentar representar XOR con el mismo perceptrón simple;
2. analizar el resultado;
3. implementar una red con una capa oculta para representar XOR.

> La idea central de la actividad es distinguir entre **optimización** —cómo se ajustan los parámetros— y **capacidad de representación** —qué funciones puede representar una arquitectura.


## Objetivos

Al finalizar la actividad debería poder:

- construir un modelo sencillo con `nn.Sequential`;
- interpretar `nn.Linear(2,1)` y su `bias`;
- realizar un *forward pass*;
- calcular una función de costo;
- usar `loss.backward()` para obtener gradientes;
- actualizar parámetros mediante descenso por gradiente;
- entrenar con `torch.optim.SGD`;
- evaluar las cuatro configuraciones binarias;
- relacionar la frontera de decisión con $h=0$;
- comparar OR y XOR;
- implementar una red $2\to2\to1$ (red con una capa oculta).


# 1. Preparación del entorno


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("Versión de PyTorch:", torch.__version__)
torch.set_printoptions(precision=4, sci_mode=False)


# 2. Datos de entrada

Para dos variables binarias existen solamente cuatro configuraciones:

$$
(0,0),\quad(0,1),\quad(1,0),\quad(1,1).
$$


In [ ]:
X = torch.tensor([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.]
])

y_or = torch.tensor([
    [0.],
    [1.],
    [1.],
    [1.]
])

print("X =")
print(X)

print("y_or =")
print(y_or)

print("\nX.shape =", X.shape)
print("y_or.shape =", y_or.shape)


## Interpretación de las dimensiones

`X.shape == (4,2)` significa:

- 4 ejemplos;
- 2 variables de entrada por ejemplo.

`y_or.shape == (4,1)` significa que existe una salida deseada por ejemplo.


# 3. Funciones auxiliares

Las siguientes funciones se utilizarán primero en el ejemplo OR y luego podrán reutilizarse en las actividades de XOR.


In [ ]:
def graficar_puntos(X, y, titulo):
    Xn = X.detach().cpu().numpy()
    yn = y.detach().cpu().numpy().ravel()

    plt.figure(figsize=(5.5, 5))

    for clase, marcador in [(0, "o"), (1, "s")]:
        mask = yn == clase
        plt.scatter(
            Xn[mask, 0],
            Xn[mask, 1],
            s=150,
            marker=marcador,
            edgecolors="black",
            label=f"Clase {clase}"
        )

    plt.xlim(-0.3, 1.3)
    plt.ylim(-0.3, 1.3)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(titulo)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


def crear_modelo_lineal(semilla=7):
    torch.manual_seed(semilla)

    return nn.Sequential(
        nn.Linear(2, 1),
        nn.Sigmoid()
    )

"""
def clases(modelo, X, umbral=0.5):
    with torch.no_grad():
        return (modelo(X) >= umbral).int()
"""

def entrenar(modelo, X, y, epocas=4000, eta=0.5):
    criterio = nn.MSELoss()
    optimizador = torch.optim.SGD(modelo.parameters(), lr=eta)

    historial = {
        "antes": [],
        "despues": []
    }

    for _ in range(epocas):
        # 1. Forward pass
        y_pred = modelo(X)

        # 2. Costo antes de actualizar
        loss_antes = 0.5 * criterio(y_pred, y)
        historial["antes"].append(loss_antes.item())

        # 3. Gradientes
        optimizador.zero_grad()
        loss_antes.backward()

        # 4. Actualización de parámetros
        optimizador.step()

        # 5. Costo después de actualizar
        with torch.no_grad():
            y_pred_despues = modelo(X)
            loss_despues = 0.5 * criterio(y_pred_despues, y)

        historial["despues"].append(loss_despues.item())

    return historial


def graficar_costos_antes_despues(historial, titulo):
    epocas = range(1, len(historial["antes"]) + 1)

    plt.figure(figsize=(7.5, 4.5))
    plt.plot(epocas, historial["antes"], label="Antes de actualizar")
    plt.plot(epocas, historial["despues"], "--", label="Después de actualizar")
    plt.xlabel("Época")
    plt.ylabel("Costo")
    plt.title(titulo)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


def graficar_frontera_lineal(modelo, X, y, titulo):
    Xn = X.detach().cpu().numpy()
    yn = y.detach().cpu().numpy().ravel()

    w = modelo[0].weight.detach().flatten()
    b = modelo[0].bias.detach().item()

    plt.figure(figsize=(5.5, 5))

    for clase, marcador in [(0, "o"), (1, "s")]:
        mask = yn == clase
        plt.scatter(
            Xn[mask, 0],
            Xn[mask, 1],
            s=150,
            marker=marcador,
            edgecolors="black",
            label=f"Clase {clase}"
        )

    # Frontera: w1*x1 + w2*x2 + b = 0
    if abs(w[1].item()) > 1e-10:
        x1 = torch.linspace(-0.3, 1.3, 200)
        x2 = -(w[0] * x1 + b) / w[1]
        plt.plot(x1.numpy(), x2.numpy(), label="Frontera h=0")
    elif abs(w[0].item()) > 1e-10:
        x1_const = -b / w[0].item()
        plt.axvline(x1_const, label="Frontera h=0")

    plt.xlim(-0.3, 1.3)
    plt.ylim(-0.3, 1.3)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(titulo)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


# 4. Ejemplo resuelto: compuerta OR

La tabla OR es:

| $x_1$ | $x_2$ | OR |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

Primero observaremos la geometría del problema.


In [ ]:
graficar_puntos(X, y_or, "Compuerta OR")


## 4.1. Construcción del *perceptron* simple

La arquitectura es:

```python
nn.Sequential(
    nn.Linear(2, 1),
    nn.Sigmoid()
)
```
```text
x1 ─────┐
        ├──> Linear(2,1, bias=True) ──> Sigmoid ──> ŷ
x2 ─────┘
```


`nn.Linear(2,1)` calcula:

$$
h=w_1x_1+w_2x_2+b.
$$

La sigmoide produce:

$$
\hat y=\sigma(h).
$$


In [ ]:
modelo_or = crear_modelo_lineal(semilla=7)
print(modelo_or)


## 4.2. Inspección de parámetros

`weight` contiene $w_1$ y $w_2$.

`bias` contiene $b$.


In [ ]:
capa_lineal = modelo_or[0]

print("Pesos [w1, w2]:")
print(capa_lineal.weight)
print("Shape de weight:", capa_lineal.weight.shape)

print("\nBias b:")
print(capa_lineal.bias)
print("Shape de bias:", capa_lineal.bias.shape)


## 4.3. *Forward pass* inicial


In [ ]:
with torch.no_grad():
    y_inicial_or = modelo_or(X)
    clases_iniciales_or = (y_inicial_or >= 0.5).int()

print("Salidas iniciales:")
print(y_inicial_or)

print("\nClases iniciales:")
print(clases_iniciales_or)


## 4.4. Función de costo

Usaremos:

$$
E=
\frac{1}{2M}
\sum_{m=1}^{M}
\left(\hat y^{(m)}-y^{(m)}\right)^2.
$$

`nn.MSELoss()` calcula el promedio del error cuadrático. Multiplicamos por $0.5$ para mantener la misma convención utilizada en la derivaciones matemáticas usuales.


In [ ]:
criterio = nn.MSELoss()

y_pred = modelo_or(X)
loss_or = 0.5 * criterio(y_pred, y_or)

print("Costo inicial para OR:", loss_or.item())


## 4.5. Cálculo de gradientes

`loss.backward()` calcula los gradientes respecto de todos los parámetros aprendibles.

Los gradientes de $w_1,w_2$ quedan en:

```python
modelo_or[0].weight.grad
```

y el gradiente de $b$ en:

```python
modelo_or[0].bias.grad
```


In [ ]:
modelo_or.zero_grad()

y_pred = modelo_or(X)
loss_or = 0.5 * criterio(y_pred, y_or)
loss_or.backward()

print("Gradientes de los pesos:")
print(modelo_or[0].weight.grad)

print("\nGradiente del bias:")
print(modelo_or[0].bias.grad)


## 4.6. Una actualización manual

El descenso por gradiente utiliza:

$$
\theta_i
\leftarrow
\theta_i-
\eta\frac{\partial E}{\partial\theta_i}.
$$

Para modificar parámetros directamente utilizamos `torch.no_grad()`.


In [ ]:
eta = 0.5

modelo_manual = crear_modelo_lineal(semilla=7)
criterio_manual = nn.MSELoss()

# Forward y backward
y_pred = modelo_manual(X)
loss_antes_manual = 0.5 * criterio_manual(y_pred, y_or)

modelo_manual.zero_grad()
loss_antes_manual.backward()

print("Costo antes:", loss_antes_manual.item())

with torch.no_grad():
    for parametro in modelo_manual.parameters():
        parametro -= eta * parametro.grad

with torch.no_grad():
    loss_despues_manual = 0.5 * criterio_manual(modelo_manual(X), y_or)

print("Costo después:", loss_despues_manual.item())


## 4.7. Entrenamiento con SGD

En cada época:

1. hacemos el *forward pass*;
2. calculamos $E_{\mathrm{antes}}$;
3. calculamos gradientes;
4. actualizamos parámetros;
5. calculamos $E_{\mathrm{después}}$.


In [ ]:
modelo_or = crear_modelo_lineal(semilla=7)

historial_or = entrenar(
    modelo_or,
    X,
    y_or,
    epocas=4000,
    eta=0.5
)

print("Costo inicial:", historial_or["antes"][0])
print("Costo final:", historial_or["despues"][-1])

graficar_costos_antes_despues(
    historial_or,
    "OR: costo antes y después de cada actualización"
)


## 4.8. Evaluación del modelo OR


In [ ]:
with torch.no_grad():
    salidas_or = modelo_or(X)
    pred_or = (salidas_or >= 0.5).int()

aciertos_or = (pred_or == y_or.int()).sum().item()

print(" x1   x2   salida      predicción   correcta")
print("------------------------------------------------")

for i in range(len(X)):
    print(
        f" {int(X[i,0])}    {int(X[i,1])}    "
        f"{salidas_or[i].item():.4f}          "
        f"{pred_or[i].item()}            "
        f"{int(y_or[i].item())}"
    )

print(f"\nAciertos: {aciertos_or}/{len(X)}")
print("Costo final:", historial_or["despues"][-1])


## 4.9. Frontera de decisión

Como:

$$
\sigma(0)=0.5,
$$

la frontera de decisión se encuentra en:

$$
h=0.
$$

Por lo tanto:

$$
w_1x_1+w_2x_2+b=0.
$$


In [ ]:
graficar_frontera_lineal(
    modelo_or,
    X,
    y_or,
    "OR: frontera de decisión aprendida"
)


# 5. Actividad guiada: XOR con el mismo perceptrón simple

Ahora deberá repetir los pasos del ejemplo OR utilizando la compuerta XOR, **sin modificar todavía la arquitectura**.


## Lista de tareas — XOR con perceptrón simple

1. Defina la tabla de verdad de XOR.
2. Grafique los cuatro puntos.
3. Cree un nuevo perceptrón simple.
4. Inspeccione los parámetros iniciales.
5. Realice el forward pass inicial.
6. Obtenga las clases usando el umbral $0.5$.
7. Calcule el costo inicial.
8. Calcule e inspeccione los gradientes.
9. Realice una actualización manual.
10. Entrene un modelo nuevo con SGD.
11. Grafique el costo antes y después de cada actualización.
12. Evalúe las cuatro configuraciones.
13. Calcule el número de aciertos.
14. Inspeccione los parámetros finales.
15. Grafique la frontera de decisión.
16. Repita el entrenamiento con distintas semillas.
17. Compare la geometría de OR y XOR.
18. Formule una conclusión sobre la capacidad de la arquitectura.


## Tarea 1 — Definir las salidas de XOR

XOR vale 1 cuando las dos entradas son diferentes.

Construya `y_xor` con `shape == (4,1)`.


In [ ]:
# TODO 1: Definir las salidas de XOR
y_xor = torch.tensor([
    [0.],
    [1.],
    [1.],
    [0.]
])

print("X =")
print(X)

print("\ny_xor =")
print(y_xor)

print("\nX.shape =", X.shape)
print("y_xor.shape =", y_xor.shape)


## Tarea 2 — Graficar XOR

Use la función `graficar_puntos(...)`.

Antes de entrenar, observe la disposición de las dos clases.


In [ ]:
# TODO 2: Graficar los puntos de la compuerta XOR
graficar_puntos(X, y_xor, "Compuerta XOR")


## Tarea 3 — Crear un nuevo perceptrón simple

No reutilice el modelo entrenado para OR.

Use la misma arquitectura:

```python
nn.Linear(2, 1)
nn.Sigmoid()
```


In [ ]:
# TODO 3: Crear nuevo modelo lineal para XOR
modelo_xor = crear_modelo_lineal(semilla=7)
print("Modelo inicial para XOR:")
print(modelo_xor)


## Tarea 4 — Inspeccionar parámetros iniciales

Imprima los pesos, el `bias` y sus `shape`.

Identifique $w_1$, $w_2$ y $b$.


In [ ]:
# TODO 4: Inspeccionar parámetros iniciales
capa_lineal_xor = modelo_xor[0]

print("Pesos iniciales [w1, w2]:")
print(capa_lineal_xor.weight)
print("Shape de weight:", capa_lineal_xor.weight.shape)

print("\nBias inicial b:")
print(capa_lineal_xor.bias)
print("Shape de bias:", capa_lineal_xor.bias.shape)

w1_ini = capa_lineal_xor.weight[0, 0].item()
w2_ini = capa_lineal_xor.weight[0, 1].item()
b_ini = capa_lineal_xor.bias[0].item()
print(f"\nValores iniciales: w1 = {w1_ini:.4f}, w2 = {w2_ini:.4f}, b = {b_ini:.4f}")


## Tarea 5 — *Forward pass* inicial

Obtenga:

- las salidas continuas;
- las clases usando el umbral $0.5$.

Compare con la tabla de verdad de XOR.


In [ ]:
# TODO 5: Forward pass inicial y clases predichas
with torch.no_grad():
    y_inicial_xor = modelo_xor(X)
    clases_iniciales_xor = (y_inicial_xor >= 0.5).int()

print("Salidas continuas iniciales (y_hat):")
print(y_inicial_xor)

print("\nClases predichas iniciales (umbral >= 0.5):")
print(clases_iniciales_xor)

print("\nValores esperados (y_xor):")
print(y_xor)

aciertos_iniciales = (clases_iniciales_xor == y_xor.int()).sum().item()
print(f"\nAciertos iniciales: {aciertos_iniciales}/{len(X)} ({aciertos_iniciales/len(X)*100:.1f}%)")


## Tarea 6 — Costo inicial

Use la misma expresión utilizada para OR:

$$
E=
\frac{1}{2M}
\sum_m
\left(\hat y^{(m)}-y^{(m)}\right)^2.
$$


In [ ]:
# TODO 6: Cálculo del costo inicial (MSE y BCELoss)
criterio_mse = nn.MSELoss()
criterio_bce = nn.BCELoss()

y_pred_xor = modelo_xor(X)
loss_xor_mse = 0.5 * criterio_mse(y_pred_xor, y_xor)
loss_xor_bce = criterio_bce(y_pred_xor, y_xor)

print("Costo inicial para XOR (0.5 * MSELoss):", loss_xor_mse.item())
print("Costo inicial para XOR (BCELoss):       ", loss_xor_bce.item())


## Tarea 7 — Gradientes

Repita:

```python
modelo_xor.zero_grad()
...
loss_xor.backward()
```

e inspeccione:

```python
modelo_xor[0].weight.grad
modelo_xor[0].bias.grad
```


In [ ]:
# TODO 7: Cálculo e inspección de gradientes
modelo_xor.zero_grad()

y_pred_xor = modelo_xor(X)
loss_xor = 0.5 * nn.MSELoss()(y_pred_xor, y_xor)
loss_xor.backward()

print("Gradientes de los pesos (dE/dw1, dE/dw2):")
print(modelo_xor[0].weight.grad)

print("\nGradiente del bias (dE/db):")
print(modelo_xor[0].bias.grad)


## Tarea 8 — Actualización manual

Actualice los parámetros dentro de:

```python
with torch.no_grad():
```

y vuelva a calcular el costo.

**Pregunta:** ¿disminuyó el costo después del paso?


In [ ]:
# TODO 8: Actualización manual con eta = 0.5
eta = 0.5

modelo_manual_xor = crear_modelo_lineal(semilla=7)
criterio_manual = nn.MSELoss()

# Forward y backward inicial
y_pred = modelo_manual_xor(X)
loss_antes_manual = 0.5 * criterio_manual(y_pred, y_xor)

modelo_manual_xor.zero_grad()
loss_antes_manual.backward()

print("Costo antes de la actualización manual:", loss_antes_manual.item())

# Actualización de parámetros: theta <- theta - eta * gradiente
with torch.no_grad():
    for parametro in modelo_manual_xor.parameters():
        parametro -= eta * parametro.grad

with torch.no_grad():
    loss_despues_manual = 0.5 * criterio_manual(modelo_manual_xor(X), y_xor)

print("Costo después de la actualización manual:", loss_despues_manual.item())
print(f"Variación (ΔE = E_después - E_antes): {loss_despues_manual.item() - loss_antes_manual.item():.6f}")
print(f"¿Disminuyó el costo?: {'Sí' if loss_despues_manual.item() < loss_antes_manual.item() else 'No'}")


## Tarea 9 — Entrenamiento con SGD

Cree un modelo nuevo y use `entrenar(...)`.

Como punto de partida:

```python
epocas=500
eta=0.5
```

Luego puede experimentar con más épocas.


In [ ]:
# TODO 9: Entrenamiento de XOR con perceptrón simple usando SGD, lr=0.1, epochs=1000 y BCELoss
def entrenar_bce(modelo, X, y, epocas=1000, eta=0.1):
    criterio = nn.BCELoss()
    optimizador = torch.optim.SGD(modelo.parameters(), lr=eta)

    historial = {
        "antes": [],
        "despues": []
    }

    for _ in range(epocas):
        # 1. Forward pass
        y_pred = modelo(X)

        # 2. Costo antes de actualizar
        loss_antes = criterio(y_pred, y)
        historial["antes"].append(loss_antes.item())

        # 3. Gradientes
        optimizador.zero_grad()
        loss_antes.backward()

        # 4. Actualización de parámetros
        optimizador.step()

        # 5. Costo después de actualizar
        with torch.no_grad():
            y_pred_despues = modelo(X)
            loss_despues = criterio(y_pred_despues, y)

        historial["despues"].append(loss_despues.item())

    return historial

# Entrenamos el modelo con SGD, lr=0.1, epochs=1000
modelo_xor = crear_modelo_lineal(semilla=7)
historial_xor = entrenar_bce(
    modelo_xor,
    X,
    y_xor,
    epocas=1000,
    eta=0.1
)

print("Entrenamiento completado para XOR (Perceptrón simple):")
print(f" - Costo inicial (época 1):   {historial_xor['antes'][0]:.6f}")
print(f" - Costo final   (época 1000): {historial_xor['despues'][-1]:.6f}")


## Tarea 10 — Graficar el costo

Analice:

- ¿disminuye al principio?
- ¿se estabiliza?
- ¿la diferencia entre $E_{\mathrm{antes}}$ y $E_{\mathrm{después}}$ se hace pequeña?


In [ ]:
# TODO 10: Graficar la evolución del costo
graficar_costos_antes_despues(
    historial_xor,
    "XOR (Perceptrón simple): Costo antes y después de cada actualización (SGD, lr=0.1)"
)

diff_final_xor = abs(historial_xor["antes"][-1] - historial_xor["despues"][-1])
print(f"Costo final: {historial_xor['despues'][-1]:.6f}")
print(f"Diferencia final entre costo antes y después: {diff_final_xor:.2e}")
print("\nAnálisis:")
print("1. ¿Disminuye al principio?: Sí, desciende ligeramente durante las primeras épocas.")
print(f"2. ¿Se estabiliza?: Sí, se estanca en una meseta en torno a {historial_xor['despues'][-1]:.4f} (≈ ln(2) = 0.6931).")
print("3. ¿La diferencia se hace pequeña?: Sí, es prácticamente nula porque el gradiente se anula al alcanzar")
print("   la mejor solución de compromiso posible para una frontera lineal, sin lograr resolver XOR.")


## Tarea 11 — Evaluación de las cuatro entradas

Muestre para cada ejemplo:

```text
x1   x2   salida   predicción   valor correcto
```

Calcule también la cantidad de aciertos.

No observe solamente las clases: examine los valores continuos $\hat y$.


In [ ]:
# TODO 11: Evaluación de las cuatro entradas sobre XOR
with torch.no_grad():
    salidas_xor = modelo_xor(X)
    pred_xor = (salidas_xor >= 0.5).int()

aciertos_xor = (pred_xor == y_xor.int()).sum().item()

print(" x1   x2   salida      predicción   valor correcto   ¿Acierto?")
print("--------------------------------------------------------------")

for i in range(len(X)):
    acerto = "Sí" if pred_xor[i].item() == int(y_xor[i].item()) else "No"
    print(
        f" {int(X[i,0])}    {int(X[i,1])}    "
        f"{salidas_xor[i].item():.4f}          "
        f"{pred_xor[i].item()}               "
        f"{int(y_xor[i].item())}             "
        f"{acerto}"
    )

print(f"\nAciertos: {aciertos_xor}/{len(X)} ({aciertos_xor/len(X)*100:.1f}%)")
print("Costo final:", historial_xor["despues"][-1])
print("\nObservación: Las salidas continuas se concentran alrededor de 0.5,")
print("evidenciando la incapacidad de separar las cuatro configuraciones simultáneamente.")


## Tarea 12 — Parámetros finales y frontera

Imprima:

$$
w_1,\quad w_2,\quad b.
$$

Luego grafique la frontera:

$$
w_1x_1+w_2x_2+b=0.
$$

**Pregunta:** ¿una única recta puede separar correctamente las dos clases de XOR?


In [ ]:
# TODO 12: Inspeccionar parámetros finales y graficar la frontera lineal
w_xor = modelo_xor[0].weight.detach().flatten()
b_xor = modelo_xor[0].bias.detach().item()

print("Parámetros finales aprendidos:")
print(f"w1 = {w_xor[0].item():.4f}")
print(f"w2 = {w_xor[1].item():.4f}")
print(f"b  = {b_xor:.4f}")

graficar_frontera_lineal(
    modelo_xor,
    X,
    y_xor,
    "XOR: frontera de decisión aprendida por el perceptrón simple"
)

print("\n¿Una única recta puede separar correctamente las dos clases de XOR?:")
print("NO. Ninguna línea recta en R^2 puede separar los puntos (0,1) y (1,0) (clase 1)")
print("de los puntos (0,0) y (1,1) (clase 0) porque se encuentran en vértices opuestos diagonalmente.")


## Tarea 13 — Distintas inicializaciones

Repita el entrenamiento con varias semillas, por ejemplo:

```python
[1, 7, 15, 25]
```

Para cada caso registre:

- costo final;
- salidas continuas;
- predicciones;
- número de aciertos.

**Pregunta:** ¿cambiar la inicialización modifica la capacidad de representación del modelo?


In [ ]:
# TODO 13: Entrenar con múltiples semillas
semillas = [1, 7, 15, 25]

print(f"{'Semilla':<8}{'Costo final':<14}{'Aciertos':<10}{'Salidas continuas':<32}{'Predicciones'}")
print("-" * 75)

for semilla in semillas:
    mod_sem = crear_modelo_lineal(semilla=semilla)
    hist_sem = entrenar_bce(mod_sem, X, y_xor, epocas=1000, eta=0.1)

    with torch.no_grad():
        out_cont = mod_sem(X).flatten()
        out_pred = (out_cont >= 0.5).int().flatten()
        aciertos_sem = (out_pred == y_xor.flatten().int()).sum().item()

    out_cont_str = "[" + ", ".join([f"{val:.2f}" for val in out_cont]) + "]"
    out_pred_str = str(out_pred.tolist())
    print(f"{semilla:<8}{hist_sem['despues'][-1]:<14.4f}{aciertos_sem:<10}{out_cont_str:<32}{out_pred_str}")

print("\nRespuesta: NO. Cambiar la inicialización modifica el punto de inicio en el espacio de parámetros")
print("y la trayectoria de optimización, pero NO altera la familia de funciones que el modelo puede")
print("representar. Una neurona lineal simple siempre producirá un hiperplano lineal.")


# 6. Análisis de la separabilidad lineal de XOR

Use la condición:

$$
h=w_1x_1+w_2x_2+b.
$$

Para clase 1 (salida $\hat{y} \geq 0.5 \iff h \geq 0$):

$$
h\geq0.
$$

Para clase 0 (salida $\hat{y} < 0.5 \iff h < 0$):

$$
h<0.
$$

Escribimos la condición requerida para cada punto:

1. **Punto $(0,0)$ — Clase 0:**
   $$w_1(0) + w_2(0) + b < 0 \implies b < 0$$

2. **Punto $(0,1)$ — Clase 1:**
   $$w_1(0) + w_2(1) + b \ge 0 \implies w_2 + b \ge 0$$

3. **Punto $(1,0)$ — Clase 1:**
   $$w_1(1) + w_2(0) + b \ge 0 \implies w_1 + b \ge 0$$

4. **Punto $(1,1)$ — Clase 0:**
   $$w_1(1) + w_2(1) + b < 0 \implies w_1 + w_2 + b < 0$$

---

### Respuestas y análisis teórico:

1. **¿Pueden satisfacerse simultáneamente las cuatro condiciones?**
   **No, es imposible.**
   Sumando las dos desigualdades de clase 1 (puntos $(0,1)$ y $(1,0)$):
   $$(w_2 + b) + (w_1 + b) \ge 0 \implies w_1 + w_2 + 2b \ge 0$$
   Como la condición para $(0,0)$ impone estrictamente $b < 0$, tenemos:
   $$w_1 + w_2 + b > w_1 + w_2 + 2b \ge 0 \implies w_1 + w_2 + b > 0$$
   Sin embargo, la condición para el punto $(1,1)$ requiere:
   $$w_1 + w_2 + b < 0$$
   Se llega a una contradicción insalvable ($0 < w_1 + w_2 + b < 0$). Por ende, ningún conjunto de parámetros $(w_1, w_2, b)$ puede resolver XOR: **la función XOR no es linealmente separable**.

2. **¿El problema observado se debe a que SGD "no funciona"?**
   **No.** El algoritmo SGD funciona a la perfección: calcula correctamente el gradiente de la función de costo y desplaza los parámetros en la dirección de máximo descenso hasta alcanzar el mínimo alcanzable en la superficie de pérdida. El problema no es de optimización, sino que **en el espacio de búsqueda definido por la arquitectura no existe ninguna combinación de pesos que resuelva el problema**.

3. **¿Aumentar el número de épocas cambia la familia de funciones que puede representar una sola neurona?**
   **No.** El número de épocas solo controla cuántas iteraciones de descenso por gradiente se realizan. La familia de funciones representables (el espacio de hipótesis $\mathcal{H}$) está fijada de antemano por la arquitectura del modelo (un hiperplano en $\mathbb{R}^2$). Entrenar por más épocas solo mantiene al modelo oscilando alrededor del mínimo de compromiso.


# Parte 2:
# *Perceptron* con una capa oculta

# 7. Actividad guiada: XOR con una capa oculta

Ahora **aumentaremos la capacidad del modelo**.

Utilizaremos una red:

$$
2\to2\to1.
$$

En PyTorch:

```python
nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid()
)
```

La primera sigmoide actúa **elemento a elemento** sobre las dos neuronas ocultas.

El esquema de entrenamiento seguirá siendo el mismo que en OR:

$$
\text{forward}
\rightarrow
\text{costo}
\rightarrow
\text{backward}
\rightarrow
\text{actualización}.
$$


## Lista de tareas — XOR con perceptrón multicapa

1. Construya la red $2\to2\to1$.
2. Imprima el modelo e identifique sus componentes.
3. Inspeccione `weight` y `bias` de las dos capas lineales.
4. Cuente los parámetros aprendibles.
5. Realice un forward pass inicial.
6. Calcule el costo inicial.
7. Calcule los gradientes de ambas capas.
8. Entrene la red con la misma función `entrenar(...)`.
9. Grafique el costo.
10. Evalúe las cuatro entradas XOR.
11. Calcule el número de aciertos.
12. Compare las salidas con las del perceptrón simple.
13. Grafique la región de decisión.
14. Compare la frontera multicapa con la frontera lineal.
15. Formule una conclusión sobre el efecto de la capa oculta.


## Tarea 14 — Construir la red multicapa

Cree `modelo_mlp`.

Puede usar una semilla para reproducibilidad.


In [ ]:
# TODO 14: Construir la red multicapa (2 -> 2 -> 1)
torch.manual_seed(42)

modelo_mlp = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid()
)

print("Arquitectura de la red multicapa (MLP 2 -> 2 -> 1):")
print(modelo_mlp)


## Tarea 15 — Inspeccionar parámetros

Las capas lineales son:

```python
modelo_mlp[0]
modelo_mlp[2]
```

Para ambas inspeccione:

- `weight`;
- `weight.shape`;
- `bias`;
- `bias.shape`.

Luego cuente todos los parámetros aprendibles.

Puede usar:

```python
sum(p.numel() for p in modelo_mlp.parameters())
```


In [ ]:
# TODO 15: Inspeccionar parámetros y contar cuántos hay
capa_oculta = modelo_mlp[0]
capa_salida = modelo_mlp[2]

print("=== Capa 1: Oculta (Linear 2 -> 2) ===")
print("Pesos W1:")
print(capa_oculta.weight)
print("Shape W1:", capa_oculta.weight.shape)
print("\nBias b1:")
print(capa_oculta.bias)
print("Shape b1:", capa_oculta.bias.shape)

print("\n=== Capa 2: Salida (Linear 2 -> 1) ===")
print("Pesos W2:")
print(capa_salida.weight)
print("Shape W2:", capa_salida.weight.shape)
print("\nBias b2:")
print(capa_salida.bias)
print("Shape b2:", capa_salida.bias.shape)

total_params = sum(p.numel() for p in modelo_mlp.parameters() if p.requires_grad)
print(f"\nTotal de parámetros aprendibles: {total_params}")
print("Desglose: (2x2 pesos + 2 bias en capa oculta) + (2x1 pesos + 1 bias en capa salida) = 6 + 3 = 9 parámetros.")


## Tarea 16 — Forward pass inicial y costo

Evalúe la red multicapa sobre las cuatro entradas XOR.

Obtenga:

- salidas continuas;
- clases con umbral $0.5$;
- costo inicial.


In [ ]:
# TODO 16: Forward pass inicial y cálculo del costo
with torch.no_grad():
    y_inicial_mlp = modelo_mlp(X)
    clases_iniciales_mlp = (y_inicial_mlp >= 0.5).int()

criterio_bce = nn.BCELoss()
loss_inicial_mlp = criterio_bce(y_inicial_mlp, y_xor)

print("Salidas continuas iniciales MLP (y_hat):")
print(y_inicial_mlp)

print("\nClases predichas iniciales (umbral >= 0.5):")
print(clases_iniciales_mlp)

print(f"\nCosto inicial (BCELoss): {loss_inicial_mlp.item():.6f}")
aciertos_ini_mlp = (clases_iniciales_mlp == y_xor.int()).sum().item()
print(f"Aciertos iniciales: {aciertos_ini_mlp}/4 ({aciertos_ini_mlp/4*100:.1f}%)")


## Tarea 17 — Gradientes de ambas capas

Después de `loss.backward()`, inspeccione:

```python
modelo_mlp[0].weight.grad
modelo_mlp[0].bias.grad
modelo_mlp[2].weight.grad
modelo_mlp[2].bias.grad
```

Observe que autograd calcula gradientes para todos los parámetros que participaron en el forward pass.


In [ ]:
# TODO 17: Calcular e inspeccionar gradientes de ambas capas
modelo_mlp.zero_grad()
y_pred_mlp = modelo_mlp(X)
loss_mlp = criterio_bce(y_pred_mlp, y_xor)
loss_mlp.backward()

print("=== Gradientes Capa Oculta ===")
print("Gradiente de W1 (modelo_mlp[0].weight.grad):")
print(modelo_mlp[0].weight.grad)
print("\nGradiente de b1 (modelo_mlp[0].bias.grad):")
print(modelo_mlp[0].bias.grad)

print("\n=== Gradientes Capa de Salida ===")
print("Gradiente de W2 (modelo_mlp[2].weight.grad):")
print(modelo_mlp[2].weight.grad)
print("\nGradiente de b2 (modelo_mlp[2].bias.grad):")
print(modelo_mlp[2].bias.grad)


## Tarea 18 — Entrenamiento

Puede reutilizar directamente `entrenar(...)`.

Como punto de partida pruebe:

```python
epocas=6000
eta=1.0
```

Si el resultado no es satisfactorio, experimente con la semilla, las épocas o la tasa de aprendizaje.


In [ ]:
# TODO 18: Entrenar la red multicapa con SGD, lr=0.5, epochs=10000 y BCELoss
torch.manual_seed(42)

modelo_mlp = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid()
)

# Entrenamos con BCELoss y optim.SGD (lr=0.5, 10000 épocas)
historial_mlp = entrenar_bce(
    modelo_mlp,
    X,
    y_xor,
    epocas=10000,
    eta=0.5
)

print("Entrenamiento completado para XOR con red multicapa (2 -> 2 -> 1):")
print(f" - Costo inicial (época 1):     {historial_mlp['antes'][0]:.6f}")
print(f" - Costo final   (época 10000): {historial_mlp['despues'][-1]:.6f}")


## Tarea 19 — Costo y evaluación

Grafique el historial y luego informe:

- las cuatro salidas continuas;
- las predicciones;
- los valores correctos;
- el número de aciertos;
- el costo final.

Compare estos resultados con el perceptrón simple.


In [ ]:
# TODO 19: Graficar costo y evaluar las cuatro entradas en MLP
graficar_costos_antes_despues(
    historial_mlp,
    "XOR (Red multicapa 2->2->1): Evolución del costo (BCELoss, lr=0.5, 10000 épocas)"
)

with torch.no_grad():
    salidas_mlp = modelo_mlp(X)
    pred_mlp = (salidas_mlp >= 0.5).int()

aciertos_mlp = (pred_mlp == y_xor.int()).sum().item()

print(" x1   x2   salida      predicción   valor correcto   ¿Acierto?")
print("--------------------------------------------------------------")

for i in range(len(X)):
    acerto = "Sí" if pred_mlp[i].item() == int(y_xor[i].item()) else "No"
    print(
        f" {int(X[i,0])}    {int(X[i,1])}    "
        f"{salidas_mlp[i].item():.4f}          "
        f"{pred_mlp[i].item()}               "
        f"{int(y_xor[i].item())}             "
        f"{acerto}"
    )

print(f"\nAciertos: {aciertos_mlp}/{len(X)} ({aciertos_mlp/len(X)*100:.1f}%)")
print(f"Costo final: {historial_mlp['despues'][-1]:.6f}")

print("\n--- Comparación con el perceptrón simple ---")
print(f"Perceptrón simple:     Aciertos = {aciertos_xor}/4 ({aciertos_xor/4*100:.1f}%), Costo final = {historial_xor['despues'][-1]:.4f}")
print(f"Red multicapa 2->2->1: Aciertos = {aciertos_mlp}/4 ({aciertos_mlp/4*100:.1f}%), Costo final = {historial_mlp['despues'][-1]:.4f}")


## Tarea 20 — Región de decisión

El color del gráfico deberá representar la salida continua:

$$
\hat y(x_1,x_2).
$$

La curva:

$$
\hat y=0.5
$$

será la frontera de decisión.

Complete la función siguiente.


In [ ]:
# TODO 20: Función para graficar la región de decisión de la red multicapa
def graficar_region_decision(modelo, X, y, titulo):
    # 1. Crear una grilla de puntos en el plano
    eje = torch.linspace(-0.3, 1.3, 200)
    xx, yy = torch.meshgrid(eje, eje, indexing="xy")
    puntos = torch.stack([xx.flatten(), yy.flatten()], dim=1)

    # 2. Evaluar el modelo sobre toda la grilla sin gradientes
    with torch.no_grad():
        zz = modelo(puntos).reshape(xx.shape)

    # 3. Preparar los puntos originales
    Xn = X.detach().cpu().numpy()
    yn = y.detach().cpu().numpy().ravel()
    xx_np = xx.numpy()
    yy_np = yy.numpy()
    zz_np = zz.numpy()

    plt.figure(figsize=(6, 5))

    # 4. Dibujar contourf para la salida continua (mapa de probabilidad)
    cont = plt.contourf(xx_np, yy_np, zz_np, levels=50, cmap="coolwarm", alpha=0.7)
    cbar = plt.colorbar(cont)
    cbar.set_label("Salida continua $\\hat{y}$", rotation=270, labelpad=15)

    # 5. Dibujar el contorno del nivel 0.5 (frontera de decisión)
    cs = plt.contour(xx_np, yy_np, zz_np, levels=[0.5], colors="black", linewidths=2.5)
    plt.clabel(cs, fmt="$\\hat{y}=0.5$", inline=True, fontsize=10)

    # 6. Superponer las dos clases originales
    for clase, marcador, color in [(0, "o", "blue"), (1, "s", "red")]:
        mask = yn == clase
        plt.scatter(
            Xn[mask, 0],
            Xn[mask, 1],
            s=160,
            marker=marcador,
            color=color,
            edgecolors="black",
            linewidths=1.5,
            label=f"Clase {clase}",
            zorder=5
        )

    # 7. Agregar ejes, título y leyenda
    plt.xlim(-0.3, 1.3)
    plt.ylim(-0.3, 1.3)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(titulo)
    plt.grid(alpha=0.3, linestyle="--")
    plt.legend(loc="upper right")
    plt.show()

# Ejecutar la función sobre el modelo multicapa
graficar_region_decision(modelo_mlp, X, y_xor, "XOR con capa oculta (2->2->1): Región y Frontera de Decisión")


# 8. Comparación final

### Tabla comparativa completa

| Aspecto | Perceptrón simple | Red con capa oculta (MLP 2→2→1) |
|---|---|---|
| **Entradas** | 2 ($x_1, x_2$) | 2 ($x_1, x_2$) |
| **Capa oculta** | No (0 neuronas) | Sí (2 neuronas con activación sigmoide) |
| **Número de parámetros** | 3 ($2 \text{ pesos } + 1 \text{ bias}$) | 9 ($(2\times 2 + 2) + (2\times 1 + 1) = 6 + 3$) |
| **Tipo de frontera en el plano original** | Lineal (una sola recta: $w_1 x_1 + w_2 x_2 + b = 0$) | No lineal (curva compuesta que separa regiones no convexas) |
| **¿Representa OR?** | **Sí** (100% de aciertos, 4/4) | **Sí** (100% de aciertos, 4/4) |
| **¿Representa XOR?** | **No** (máximo 75% o 50% de aciertos, error no nulo) | **Sí** (100% de aciertos, 4/4, pérdida $\to 0$) |

---

## Análisis comparativo: Optimización vs. Capacidad de Representación

### 1. Comparación de la función de pérdida durante el entrenamiento
- **Perceptrón simple:** La pérdida desciende levemente en las primeras épocas y luego entra en una meseta rígida en torno a $\ln(2) \approx 0.6931$ (para BCELoss) o $0.125$ (para MSELoss). La diferencia entre $E_{\mathrm{antes}}$ y $E_{\mathrm{después}}$ converge rápidamente a cero: el gradiente se anula porque la red se halla en el mínimo del espacio de hipótesis lineal disponible. Al no poder doblar la frontera, predecir $\approx 0.5$ es la mejor solución de compromiso global para minimizar la pérdida.
- **Red con capa oculta (2→2→1):** La curva de pérdida desciende de forma consistente hacia cero ($< 0.01$). Gracias a la flexibilidad no lineal de la capa intermedia, el descenso por gradiente puede modificar los hiperplanos internos hasta desacoplar los puntos y alcanzar una clasificación perfecta con error despreciable.

### 2. Discusión: Optimización vs. Capacidad de Representación
- **Capacidad de representación (Espacio de hipótesis $\mathcal{H}$):** Refleja qué clase de funciones matemáticas puede computar una arquitectura fija. Para el perceptrón simple, $\mathcal{H}$ se limita a hiperplanos lineales en $\mathbb{R}^2$. Dado que XOR no es linealmente separable, la función XOR **no pertenece a $\mathcal{H}$**.
- **Optimización (SGD):** Es el algoritmo numérico encargado de buscar los mejores parámetros dentro de $\mathcal{H}$. El fracaso del perceptrón simple no se debe a que SGD falle; SGD encuentra el punto óptimo dentro de $\mathcal{H}$, pero dicho óptimo es insuficiente por limitaciones geométricas intrínsecas de la arquitectura.
- **Efecto de la capa oculta:** Al incorporar 2 neuronas ocultas con funciones de activación sigmoide, la red realiza un mapeo no lineal $\mathbf{x} \mapsto \mathbf{h} = \sigma(W_1 \mathbf{x} + \mathbf{b}_1)$. En el espacio latente $(h_1, h_2)$, los puntos de XOR se transforman en una configuración linealmente separable, la cual puede ser clasificada trivialmente por la última neurona lineal.

---

## Preguntas de cierre

1. **¿El descenso por gradiente consiguió disminuir el costo en el perceptrón simple aplicado a XOR?**
   **Sí.** Durante las primeras épocas el costo se redujo respecto de su valor aleatorio inicial, alcanzando un mínimo local/global de la superficie de pérdida de esa arquitectura.

2. **¿Por qué eso no implica necesariamente una clasificación perfecta?**
   Porque la optimización solo minimiza el error *dentro del espacio de funciones que el modelo puede generar*. Si el modelo solo puede trazar una línea recta, la mejor línea posible deja inevitablemente al menos un punto mal clasificado (o predice $\approx 0.5$ para todos), con una precisión máxima de $75\%$ o $50\%$.

3. **¿Por qué más épocas no solucionan una limitación de representación?**
   Porque más épocas solo permiten dar más pasos sobre la misma superficie de pérdida. Una vez que el gradiente se anula en el mínimo alcanzable por un separador lineal, iterar más veces no altera la estructura geométrica del modelo ni añade dimensiones para doblar la frontera.

4. **¿Qué aporta la capa oculta?**
   La capa oculta aporta una **transformación no lineal del espacio de características**. Las dos neuronas ocultas actúan como detectores de características intermedias (una operando como una frontera similar a OR y otra a NAND), proyectando los datos a un espacio donde la clase 1 y la clase 0 son linealmente separables por la neurona de salida.

5. **¿Qué papel cumple la no linealidad introducida por `nn.Sigmoid()`?**
   La no linealidad es indispensable. Si no existiera `nn.Sigmoid()`, la composición de dos capas lineales $W_2(W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = (W_2 W_1)\mathbf{x} + (W_2\mathbf{b}_1 + \mathbf{b}_2)$ colapsaría exactamente en una sola transformación lineal equivalente, conservando la limitación de separabilidad lineal. La sigmoide rompe la linealidad y permite deformar el espacio.

6. **¿Qué diferencias observa entre la frontera lineal y la región de decisión del modelo multicapa?**
   La frontera lineal del perceptrón simple es una recta rígida que corta el cuadrado $[0,1]^2$ sin poder aislar las diagonales. En cambio, la región de decisión del modelo multicapa presenta una frontera no lineal curva (o franja diagonal) que encierra con precisión a los puntos $(0,1)$ y $(1,0)$ en la región positiva (roja) y a $(0,0)$ y $(1,1)$ en la región negativa (azul), logrando un $100\%$ de aciertos.


In [ ]:
# Comparación directa de la evolución de la pérdida: Perceptrón Simple vs Red Multicapa
plt.figure(figsize=(8.5, 4.8))
plt.plot(historial_xor["despues"], label="Perceptrón Simple (lr=0.1, 1000 épocas)", color="#d62728", lw=2)
plt.plot(historial_mlp["despues"][:1000], label="Red Multicapa 2->2->1 (lr=0.5, primeras 1000 épocas)", color="#2ca02c", lw=2)
plt.plot(
    [0, 1000], [historial_mlp["despues"][-1], historial_mlp["despues"][-1]],
    "--", color="#1f77b4", label=f"Red Multicapa costo final (época 10000): {historial_mlp['despues'][-1]:.4f}"
)

plt.xlabel("Época", fontsize=11)
plt.ylabel("Costo (BCELoss)", fontsize=11)
plt.title("Comparación de la Función de Pérdida: Perceptrón Simple vs Red Multicapa", fontsize=12)
plt.grid(alpha=0.3, linestyle="--")
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()
